# Time-series forecasting

Adnan Masood · University of South Florida

Forecast 28 future days from a fictional daily series that rises over time, repeats a weekly pattern, and has random noise. Compare the forecast with what actually happened on those same dates. The generated values use teaching units; they are not a bank’s real cash balance or a production forecast.

**Dataset:** The notebook generates 365 fictional daily values beginning January 1, 2025. A steady trend adds about 1.1 units per day, a repeating seven-day pattern adds seasonal movement, and random noise makes days vary around that pattern. Seed 4 makes the series repeatable. The first 337 days are training history; the final 28 are held back to test a forecast made without seeing them.

**Task:** Fit SARIMAX(1,1,1) with a seven-day seasonal pattern, then forecast the 28 held-back dates. MAE (mean absolute error) averages the size of the forecast misses in the series’ units; lower is better on the same dates. Also count how many actual days fall inside the model’s nominal 95% prediction intervals. The 95% is a target across repeated forecasts, not a promise for these 28 days.

**Compare:** Compare with a seasonal-naive forecast: repeat the last seven values observed during training to fill the next four weeks. Both methods predict the same 28 dates, so their MAE uses the same actual values. Reusing the last week is a simple reference, not a learned model.

**Runtime:** The notebook uses NumPy to generate the series, pandas to attach dates, statsmodels for SARIMAX, and scikit-learn to calculate MAE. Fitting stops after 50 optimizer iterations; a convergence warning means the numerical search may not have settled, so check it before interpreting the output. Twenty-eight test days provide only a small check of 95% interval coverage.


In [ ]:
import numpy as np, pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
# Seed 4 makes this fictional trend, weekly cycle, and noise repeatable.
rng = np.random.default_rng(4)
t = np.arange(365)
y = 1000 + 1.1*t + 130*np.sin(2*np.pi*t/7) + rng.normal(0,45,365)
y = pd.Series(y,index=pd.date_range("2025-01-01",periods=365))
# Hold the final 28 dates back; fit using only earlier observations.
train,test = y.iloc[:-28],y.iloc[-28:]
model = SARIMAX(train,order=(1,1,1),seasonal_order=(1,0,1,7),
    enforce_stationarity=False).fit(disp=False,maxiter=50)
forecast = model.get_forecast(steps=28)
pred = forecast.predicted_mean
# MAE is the average absolute miss, in the series' original units.
print("MAE:", mean_absolute_error(test,pred))
# A simple reference repeats the last full training week four times.
seasonal_naive = np.tile(train.iloc[-7:].to_numpy(),4)
print("Seasonal naive MAE:",mean_absolute_error(test,seasonal_naive))
ci = forecast.conf_int().to_numpy()
# Coverage counts the share of test values inside their forecast intervals.
print("Interval coverage:", ((test.to_numpy()>=ci[:,0]) & (test.to_numpy()<=ci[:,1])).mean())
print(forecast.conf_int().tail())


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Compare nonseasonal order (2,0,1)

Keep seasonal_order=(1,0,1,7) and the 28-day test period; inspect MAE, coverage, and convergence warnings.

Replace:
```python
order=(1,1,1)
```
With:
```python
order=(2,0,1)
```

### Double noise standard deviation

Compare MAE, interval endpoints, and covered days out of 28.

Replace:
```python
normal(0,45,365)
```
With:
```python
normal(0,90,365)
```



In [ ]:
# Change order=(1,1,1) to order=(2,0,1), keeping the seasonal order unchanged.
# Compare both models' MAE and coverage with the same seasonal-naive forecast.


## Break it

Double the random-noise size from 45 to 90 while keeping the generated trend, weekly pattern, and seed fixed. Rerun and compare MAE and interval coverage. A wider interval may include more actual days, but it is less precise; inspect the lower and upper endpoints as well as the coverage count.


In [ ]:
# Set rng.normal(0,90,365), rerun, and compare MAE and interval widths.
# Count covered days out of 28 rather than reporting the proportion alone.


## What to submit

- SARIMAX and seasonal-naive MAE in the same units, each measured over the same 28 held-back dates.
- The number of actual days inside the model’s 95% intervals, out of 28, plus several interval widths to show how much uncertainty they allow.
- The changed-order or noise experiment, noting whether the model converged and why this short test does not establish performance across future periods.

## Optional extension

Test several forecast start dates by moving forward through time: fit only on history available at each start, predict the next period, and compare with what later happened. This walk-forward check shows whether one 28-day result repeats across seasons and market conditions.
